## Pipeline de géolocalisation des copublications Inria
### Version complète avec Nominatim (OpenStreetMap)

**Stratégie en cascade :**
1. Fusion par ID Aurehal (dictionnaire existant — aucune requête réseau)
2. Extraction de ville depuis les crochets `[Ville]` dans le nom d'organisme
3. Déduction du code État US depuis l'adresse
4. **Géocodage Nominatim** (adresse complète → OSM → lat/lon + ville) sur tous les cas sans coordonnées
5. Standardisation : une ville = une coordonnée unique (ville la plus peuplée via Geonames)
6. Nettoyage final, normalisation des caractères, export

**Pourquoi Nominatim plutôt que le scan Geonames ?**
- Envoie l'adresse **complète** → résout l'ambiguïté directement (fini Berlin/Munich/Brême pour la même ligne)
- Comprend les codes postaux, noms d'université, adresses partielles
- Gratuit, sans clé API
- Contrainte : **1 requête/seconde max** (politique OSM obligatoire)

**Pré-requis :** `pip install geopy tqdm pandas openpyxl unidecode`

In [1]:
# =========================================================
#  MASTER CONFIGURATION
# =========================================================

# --- 1. CHEMINS DES FICHIERS ---
FILE_MAIN    = "copublications_Inria_2018-2024_sans_villes.xlsx"
FILE_REF_ID  = "ID_Aurehal_Ville_Etat_Latitude_Longitude.xlsx"
FILE_CITIES  = "../cities500/cities500.txt"   # utilisé pour la standardisation finale
FILE_FINAL   = "copublications_Inria_2018-2024_avec_villes.xlsx"
FILE_FOUND   = "Resultat_Villes_TROUVEES.xlsx"
FILE_MISSING = "Resultat_Villes_MANQUANTES.xlsx"

# --- 2. COLONNES DU FICHIER PRINCIPAL ---
COL_MAIN_ID      = "id_Aurehal_org_copubliant"
COL_MAIN_ADDR    = "Adresse_org_Top_copubliant"
COL_MAIN_COUNTRY = "Nom_Pays_org_copubliant"
COL_MAIN_ORG     = "Nom_org_Top_copubliant"

# --- 3. COLONNES DU FICHIER DE RÉFÉRENCE & CIBLES ---
COL_REF_ID         = "id_Aurehal_org_copubliant"
COL_REF_LAT        = "Latitude"
COL_REF_LON        = "Longitude"
COL_REF_CITY       = "Ville"
COL_REF_STATE_SRC  = "StateCode"
COL_INTERNAL_STATE = "Statecode"

# --- 4. CONFIGURATION NOMINATIM ---
# Respecte OBLIGATOIREMENT la politique OSM : 1 requête/seconde maximum
NOMINATIM_DELAY   = 1.1    # secondes entre chaque requête
NOMINATIM_AGENT   = "inria_copublis_research"  # identifiant unique obligatoire
NOMINATIM_TIMEOUT = 10     # secondes max par requête

print("✅ Configuration chargée.")

✅ Configuration chargée.


In [2]:
# =========================================================
#  IMPORTS & FONCTIONS UTILITAIRES
# =========================================================
import pandas as pd
import re
import time
import unidecode
from tqdm import tqdm
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

tqdm.pandas()

# --- Mapping États US ---
US_STATES_MAP = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR", "CALIFORNIA": "CA",
    "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE", "FLORIDA": "FL", "GEORGIA": "GA",
    "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL", "INDIANA": "IN", "IOWA": "IA",
    "KANSAS": "KS", "KENTUCKY": "KY", "LOUISIANA": "LA", "MAINE": "ME", "MARYLAND": "MD",
    "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN", "MISSISSIPPI": "MS", "MISSOURI": "MO",
    "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV", "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ",
    "NEW MEXICO": "NM", "NEW YORK": "NY", "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH",
    "OKLAHOMA": "OK", "OREGON": "OR", "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD", "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT", "VERMONT": "VT",
    "VIRGINIA": "VA", "WASHINGTON": "WA", "WEST VIRGINIA": "WV", "WISCONSIN": "WI", "WYOMING": "WY",
    "DISTRICT OF COLUMBIA": "DC"
}
VALID_US_CODES = set(US_STATES_MAP.values())
STOPWORDS = {"la", "le", "de", "del", "da", "di", "el", "univ", "university", "institute"}

# --- Mapping pays texte → code ISO ---
COUNTRY_MAP = {
    "united states": "US", "usa": "US", "etats-unis": "US", "u.s.a.": "US",
    "united kingdom": "GB", "uk": "GB", "royaume-uni": "GB", "great britain": "GB",
    "england": "GB", "scotland": "GB", "wales": "GB",
    "france": "FR", "germany": "DE", "allemagne": "DE", "deutschland": "DE",
    "spain": "ES", "espagne": "ES", "italy": "IT", "italie": "IT",
    "india": "IN", "inde": "IN", "china": "CN", "chine": "CN", "p.r. china": "CN",
    "canada": "CA", "australia": "AU", "brazil": "BR", "bresil": "BR",
    "netherlands": "NL", "pays-bas": "NL", "holland": "NL",
    "switzerland": "CH", "suisse": "CH", "belgium": "BE", "belgique": "BE",
    "sweden": "SE", "suede": "SE", "norway": "NO", "finland": "FI", "denmark": "DK",
    "japan": "JP", "japon": "JP", "russia": "RU", "russie": "RU", "russian federation": "RU",
    "south korea": "KR", "republic of korea": "KR",
    "austria": "AT", "autriche": "AT", "ireland": "IE", "irlande": "IE",
    "czech republic": "CZ", "czechia": "CZ", "republique tcheque": "CZ",
    "portugal": "PT", "poland": "PL", "pologne": "PL",
    "israel": "IL", "greece": "GR", "grece": "GR",
    "singapore": "SG", "singapour": "SG",
    "south africa": "ZA", "afrique du sud": "ZA",
    "argentina": "AR", "argentine": "AR", "chile": "CL", "chili": "CL",
    "colombia": "CO", "colombie": "CO", "mexico": "MX", "mexique": "MX"
}


def normalize_text(text):
    if pd.isna(text): return ""
    return unidecode.unidecode(str(text)).lower().strip()


def get_country_code(pays_raw):
    if pd.isna(pays_raw): return None
    s = normalize_text(pays_raw)
    if len(s) == 2: return s.upper()
    return COUNTRY_MAP.get(s)


def fix_camel_case(text):
    if pd.isna(text): return text
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', str(text))


def extract_us_state(address):
    if pd.isna(address): return None
    clean = unidecode.unidecode(str(address)).replace(",", " ").replace(".", " ").upper()
    tokens = clean.split()
    for token in reversed(tokens):
        if token in VALID_US_CODES:
            return token
    for state_name, state_code in US_STATES_MAP.items():
        if re.search(r'\b' + re.escape(state_name) + r'\b', clean):
            return state_code
    return None


def extract_and_clean_brackets(row, col_org):
    org_name = row[col_org]
    if pd.isna(org_name): return None, org_name
    m = re.search(r"\[(.*?)\]", str(org_name))
    city_found  = None
    org_cleaned = org_name
    if m:
        content      = m.group(1).strip()
        norm_content = normalize_text(content)
        if len(norm_content) >= 3 and norm_content not in STOPWORDS:
            city_found  = content
            org_cleaned = re.sub(r"\s+", " ", re.sub(r"\[.*?\]", "", str(org_name)).strip())
    return city_found, org_cleaned


print("✅ Imports et fonctions utilitaires chargés.")

✅ Imports et fonctions utilitaires chargés.


In [3]:
# =========================================================
#  ÉTAPE 1 — FUSION PAR ID AUREHAL
#  Récupère les coordonnées déjà connues dans le dictionnaire.
#  Aucune requête réseau ici — source la plus fiable.
# =========================================================
print("--- Chargement des fichiers ---")
df_main  = pd.read_excel(FILE_MAIN)
df_right = pd.read_excel(FILE_REF_ID)
print(f"   Fichier principal  : {len(df_main)} lignes")
print(f"   Référentiel Aurehal: {len(df_right)} lignes")

if COL_REF_STATE_SRC in df_right.columns and COL_INTERNAL_STATE not in df_right.columns:
    df_right = df_right.rename(columns={COL_REF_STATE_SRC: COL_INTERNAL_STATE})

target_cols   = [COL_REF_ID, COL_REF_LAT, COL_REF_LON, COL_REF_CITY, COL_INTERNAL_STATE]
existing_cols = [c for c in target_cols if c in df_right.columns]

print("--- Fusion par ID Aurehal ---")
df = df_main.merge(
    df_right[existing_cols], how="left",
    left_on=COL_MAIN_ID, right_on=COL_REF_ID,
    suffixes=("", "_ref")
)

for col in [COL_REF_LAT, COL_REF_LON, COL_REF_CITY, COL_INTERNAL_STATE]:
    col_ref = f"{col}_ref"
    if col in df.columns and col_ref in df.columns:
        df[col] = df[col].fillna(df[col_ref])
    elif col not in df.columns and col_ref in df.columns:
        df[col] = df[col_ref]

df.drop(columns=[c for c in df.columns if c.endswith("_ref")], inplace=True)

n_aurehal = int(df[COL_REF_LAT].notna().sum())
print(f"✅ Étape 1 : {n_aurehal} lignes géolocalisées via Aurehal ({len(df) - n_aurehal} restantes)")

--- Chargement des fichiers ---
   Fichier principal  : 79530 lignes
   Référentiel Aurehal: 7576 lignes
--- Fusion par ID Aurehal ---
✅ Étape 1 : 79513 lignes géolocalisées via Aurehal (17 restantes)


In [4]:
# =========================================================
#  ÉTAPE 2 — PRÉPARATION : CROCHETS + ÉTATS US
#  Enrichit les colonnes Ville et Statecode AVANT d'appeler
#  Nominatim pour améliorer la qualité des requêtes.
# =========================================================

# 2a. Normalisation CamelCase dans les adresses
print("--- Normalisation des adresses ---")
if COL_MAIN_ADDR in df.columns:
    df[COL_MAIN_ADDR] = df[COL_MAIN_ADDR].progress_apply(fix_camel_case)

# 2b. Code État US
print("--- Extraction codes États US ---")
if COL_INTERNAL_STATE not in df.columns:
    df[COL_INTERNAL_STATE] = None

if COL_MAIN_COUNTRY in df.columns:
    mask_usa = df[COL_MAIN_COUNTRY].astype(str).str.lower().str.contains(
        r"united states|usa|etats-unis|u\.s\.a\.", na=False, regex=True
    )
else:
    mask_usa = df[COL_MAIN_ADDR].notna()

mask_us_todo = mask_usa & df[COL_MAIN_ADDR].notna() & df[COL_INTERNAL_STATE].isna()
print(f"   Extraction état sur {mask_us_todo.sum()} lignes US...")
if mask_us_todo.sum() > 0:
    df.loc[mask_us_todo, COL_INTERNAL_STATE] = (
        df.loc[mask_us_todo, COL_MAIN_ADDR].progress_apply(extract_us_state)
    )

# 2c. Villes dans les crochets [Ville] du nom d'organisme
print("--- Extraction villes depuis crochets ---")
if COL_MAIN_ORG in df.columns:
    df[["City_bracket", "Organisme_clean"]] = df.apply(
        lambda x: pd.Series(extract_and_clean_brackets(x, COL_MAIN_ORG)), axis=1
    )
    if COL_REF_CITY not in df.columns:
        df[COL_REF_CITY] = None
    df[COL_REF_CITY] = df[COL_REF_CITY].fillna(df["City_bracket"])
    df[COL_MAIN_ORG] = df["Organisme_clean"]
    df.drop(columns=["City_bracket", "Organisme_clean"], inplace=True)

n_coords = int(df[COL_REF_LAT].notna().sum())
n_todo   = len(df) - n_coords
print(f"✅ Étape 2 : {n_coords} lignes déjà géolocalisées, {n_todo} à traiter par Nominatim")

--- Normalisation des adresses ---


100%|██████████| 79530/79530 [00:00<00:00, 180789.06it/s]


--- Extraction codes États US ---
   Extraction état sur 55 lignes US...


100%|██████████| 55/55 [00:00<00:00, 6777.73it/s]


--- Extraction villes depuis crochets ---
✅ Étape 2 : 79513 lignes déjà géolocalisées, 17 à traiter par Nominatim


In [5]:
# =========================================================
#  ÉTAPE 3 — GÉOCODAGE NOMINATIM
#
#  Pour chaque ID Aurehal sans coordonnées, on essaie
#  plusieurs requêtes en cascade :
#    1. Adresse complète + pays      (le plus précis)
#    2. Adresse seule
#    3. Nom d'organisme + pays
#    4. Ville extraite (crochets) + pays
#
#  CACHE : 1 ID Aurehal unique = 1 requête maximum,
#  même si cet ID apparaît sur 500 lignes.
#
#  Durée estimée : ~1.1 sec × nb_IDs_uniques
# =========================================================

geolocator = Nominatim(user_agent=NOMINATIM_AGENT, timeout=NOMINATIM_TIMEOUT)

METHOD_LABELS = [
    "Nominatim: Adresse+Pays",
    "Nominatim: Adresse seule",
    "Nominatim: Organisme+Pays",
    "Nominatim: Ville+Pays",
]


def build_queries(row):
    """Construit les requêtes texte en ordre décroissant de précision."""
    pays  = str(row.get(COL_MAIN_COUNTRY, "") or "").strip()
    addr  = str(row.get(COL_MAIN_ADDR,    "") or "").strip()
    org   = str(row.get(COL_MAIN_ORG,     "") or "").strip()
    ville = str(row.get(COL_REF_CITY,     "") or "").strip()
    queries = []
    if addr and pays:                    queries.append(f"{addr}, {pays}")
    if addr:                             queries.append(addr)
    if org and pays and len(org) > 5:    queries.append(f"{org}, {pays}")
    if ville and pays:                   queries.append(f"{ville}, {pays}")
    # La recherche par pays uniquement a été retirée d'ici
    return queries


def geocode_with_retry(query, retries=2):
    """Appel Nominatim avec retry automatique sur timeout."""
    for attempt in range(retries + 1):
        try:
            time.sleep(NOMINATIM_DELAY)
            return geolocator.geocode(query, language="en", addressdetails=True)
        except GeocoderTimedOut:
            if attempt < retries:
                time.sleep(3)
        except GeocoderServiceError:
            return None
    return None


def extract_city_from_nominatim(location):
    """Extrait le nom de ville depuis les détails d'adresse Nominatim."""
    if location is None: return None
    addr = location.raw.get("address", {})
    for key in ["city", "town", "village", "municipality", "county", "state"]:
        val = addr.get(key)
        if val: return val
    return None


# --- Construction du cache par ID Aurehal unique ---
mask_missing   = df[COL_REF_LAT].isna()
ids_to_geocode = df.loc[mask_missing, COL_MAIN_ID].dropna().unique()

print(f"--- Géocodage Nominatim ---")
print(f"   {mask_missing.sum()} lignes sans coordonnées")
print(f"   {len(ids_to_geocode)} IDs Aurehal uniques à traiter")
print(f"   Durée estimée : ~{len(ids_to_geocode) * NOMINATIM_DELAY / 60:.0f} min")
print()

# Une ligne représentative par ID (pour construire la requête)
df_reps = (
    df[mask_missing & df[COL_MAIN_ID].notna()]
    .drop_duplicates(subset=[COL_MAIN_ID], keep="first")
    .set_index(COL_MAIN_ID)
)

# Cache : id_aurehal → (lat, lon, ville, methode)
geocode_cache = {}
stats = {"aurehal": n_aurehal, # Assurez-vous que n_aurehal est défini plus haut dans votre script
         "nominatim_addr": 0, "nominatim_org": 0,
         "nominatim_ville": 0, "echec": 0}

for aurehal_id in tqdm(ids_to_geocode, desc="Nominatim"):
    if aurehal_id not in df_reps.index:
        geocode_cache[aurehal_id] = (None, None, None, "Echec: ID absent")
        stats["echec"] += 1
        continue

    row     = df_reps.loc[aurehal_id]
    queries = build_queries(row)
    result  = None
    methode = "Echec: Introuvable"

    for i, query in enumerate(queries):
        location = geocode_with_retry(query)
        if location:
            result  = location
            methode = METHOD_LABELS[i] if i < len(METHOD_LABELS) else "Nominatim: Autre"
            break

    if result:
        ville_found = extract_city_from_nominatim(result)
        geocode_cache[aurehal_id] = (result.latitude, result.longitude, ville_found, methode)
        if "Adresse"   in methode: stats["nominatim_addr"]  += 1
        elif "Organisme" in methode: stats["nominatim_org"] += 1
        elif "Ville"     in methode: stats["nominatim_ville"] += 1
    else:
        geocode_cache[aurehal_id] = (None, None, None, "Echec: Introuvable")
        stats["echec"] += 1

print("\n✅ Géocodage Nominatim terminé.")

--- Géocodage Nominatim ---
   17 lignes sans coordonnées
   7 IDs Aurehal uniques à traiter
   Durée estimée : ~0 min



Nominatim: 100%|██████████| 7/7 [00:41<00:00,  5.99s/it]


✅ Géocodage Nominatim terminé.


In [7]:
# =========================================================
#  ÉTAPE 4 — APPLICATION DU CACHE SUR TOUTES LES LIGNES
#  Un ID Aurehal peut apparaître sur N lignes.
#  On propage les coordonnées Nominatim à toutes ces lignes.
# =========================================================
print("--- Application des résultats sur toutes les lignes ---")

if "Methode_Recherche" not in df.columns:
    df["Methode_Recherche"] = None

# Marquer les lignes déjà résolues par Aurehal
df.loc[df[COL_REF_LAT].notna() & df["Methode_Recherche"].isna(),
       "Methode_Recherche"] = "Aurehal: Deja present"

# Appliquer le cache sur les lignes encore sans coordonnées
mask_apply = df[COL_REF_LAT].isna() & df[COL_MAIN_ID].isin(geocode_cache.keys())

def apply_nominatim_result(row):
    aid = row[COL_MAIN_ID]
    if pd.isna(aid) or aid not in geocode_cache:
        return pd.Series([row.get(COL_REF_LAT), row.get(COL_REF_LON),
                          row.get(COL_REF_CITY), row.get("Methode_Recherche")])
    lat, lon, ville_nom, methode = geocode_cache[aid]
    # Conserver la ville des crochets si elle existe déjà
    city_final = row.get(COL_REF_CITY) if pd.notna(row.get(COL_REF_CITY)) else ville_nom
    return pd.Series([lat, lon, city_final, methode])

df.loc[mask_apply, [COL_REF_LAT, COL_REF_LON, COL_REF_CITY, "Methode_Recherche"]] = (
    df[mask_apply].apply(apply_nominatim_result, axis=1).values
)

n_found   = int(df[COL_REF_LAT].notna().sum())
n_missing = len(df) - n_found

print(f"\n{'='*45}")
print(f"🔹 Aurehal (dictionnaire)        : {stats['aurehal']}")
print(f"🔹 Nominatim (Adresse+Pays)      : {stats['nominatim_addr']}")
print(f"🔹 Nominatim (Organisme+Pays)    : {stats['nominatim_org']}")
print(f"🔹 Nominatim (Ville+Pays)        : {stats['nominatim_ville']}")

print(f"🔻 Toujours en échec             : {stats['echec']}")
print(f"{'='*45}")
print(f"\n✅ Total géolocalisées  : {n_found} / {len(df)}")
print(f"❌ Sans coordonnées     : {n_missing}")

--- Application des résultats sur toutes les lignes ---

🔹 Aurehal (dictionnaire)        : 79513
🔹 Nominatim (Adresse+Pays)      : 0
🔹 Nominatim (Organisme+Pays)    : 0
🔹 Nominatim (Ville+Pays)        : 0
🔻 Toujours en échec             : 7

✅ Total géolocalisées  : 79513 / 79530
❌ Sans coordonnées     : 17


In [8]:
# =========================================================
#  ÉTAPE 5 — STANDARDISATION DES COORDONNÉES
#
#  Nominatim renvoie des coordonnées très précises (bâtiment,
#  campus, rue). Pour le dashboard cartographique, on
#  normalise chaque ville à un point unique = la ville
#  la plus peuplée du même nom dans le même pays.
#
#  Source : cities500.txt (Geonames), trié population desc.
# =========================================================
print("--- Chargement Geonames pour standardisation ---")

df_geo_ref = pd.read_csv(
    FILE_CITIES, sep='\t', comment='#',
    usecols=[1, 4, 5, 8, 10, 14],
    names=['name', 'latitude', 'longitude', 'country_code', 'admin1_code', 'population'],
    dtype=str
)
df_geo_ref['latitude']   = pd.to_numeric(df_geo_ref['latitude'],   errors='coerce')
df_geo_ref['longitude']  = pd.to_numeric(df_geo_ref['longitude'],  errors='coerce')
df_geo_ref['population'] = pd.to_numeric(df_geo_ref['population'], errors='coerce').fillna(0)
df_geo_ref = df_geo_ref.sort_values('population', ascending=False)

dict_geo_simple = {}  # (nom_lower, pays_upper) → (lat, lon)
dict_geo_state  = {}  # (nom_lower, pays_upper, etat_upper) → (lat, lon)

print("   Indexation (tri population décroissant)...")
for row in tqdm(df_geo_ref.itertuples(index=False), total=len(df_geo_ref)):
    nom   = str(row.name).strip().lower()
    pays  = str(row.country_code).strip().upper()
    coord = (row.latitude, row.longitude)
    if (nom, pays) not in dict_geo_simple:
        dict_geo_simple[(nom, pays)] = coord
    if pd.notna(row.admin1_code):
        etat = str(row.admin1_code).strip().upper()
        if (nom, pays, etat) not in dict_geo_state:
            dict_geo_state[(nom, pays, etat)] = coord


def standardize_coords(row):
    ville = str(row.get(COL_REF_CITY, "") or "").strip().lower()
    pays  = get_country_code(row.get(COL_MAIN_COUNTRY))
    etat  = str(row.get(COL_INTERNAL_STATE, "") or "").strip().upper()

    if not ville or not pays:
        return row[COL_REF_LAT], row[COL_REF_LON]

    # Priorité : avec état (utile pour US, CA, AU)
    if etat:
        res = dict_geo_state.get((ville, pays, etat))
        if res: return res

    res = dict_geo_simple.get((ville, pays))
    if res: return res

    # Fallback : coordonnées Nominatim originales
    return row[COL_REF_LAT], row[COL_REF_LON]


print("   Application sur le dataframe...")
df[[COL_REF_LAT, COL_REF_LON]] = df.progress_apply(
    standardize_coords, axis=1, result_type='expand'
)

print("✅ Standardisation terminée — chaque ville a une coordonnée unique.")

--- Chargement Geonames pour standardisation ---
   Indexation (tri population décroissant)...


100%|██████████| 226173/226173 [00:01<00:00, 176226.56it/s]


   Application sur le dataframe...


100%|██████████| 79530/79530 [00:02<00:00, 30340.85it/s]

✅ Standardisation terminée — chaque ville a une coordonnée unique.


In [9]:
# =========================================================
#  ÉTAPE 6 — RENOMMAGE, NETTOYAGE FINAL & EXPORT
# =========================================================
import unidecode 

# Code pays ISO
if COL_MAIN_COUNTRY in df.columns:
    df["Code_Pays"] = df[COL_MAIN_COUNTRY].apply(get_country_code)

# Renommage des colonnes
nouvelles_colonnes = {
    'Centre_inria'              : 'Centre',
    'Equipe_inria'              : 'Equipe',
    'Auteur_Inria'              : 'Auteurs_FR',
    'Auteur_etranger'           : 'Auteurs_copubliants',
    'Nom_org_copubliant'        : 'Organisme_copubliant',
    'id_Aurehal_org_copubliant' : 'ID_Aurehal',
    'UE/Hors_UE'                : 'UE/Non_UE',
    'Annee'                     : 'Année',
    'Hal_ID'                    : 'HalID',
    'Domaine_inria'             : 'Domaine(s)',
    'Mots_cles_inria'           : 'Mots-cles',
    'Resume'                    : 'Resume',
    'Ville'                     : 'Ville',
    'Nom_Pays_org_copubliant'   : 'Pays',
    'Code_Pays'                 : 'Code_Pays',
    'Statecode'                 : 'Code_Etat',
    'Latitude'                  : 'Latitude',
    'Longitude'                 : 'Longitude',
}

cols_to_rename = {k: v for k, v in nouvelles_colonnes.items() if k in df.columns}
dashboard_df   = df.rename(columns=cols_to_rename)
cols_to_keep   = [v for v in nouvelles_colonnes.values() if v in dashboard_df.columns]
dashboard_df   = dashboard_df[cols_to_keep]

# --- Fonctions de nettoyage ---
pattern_exotiques = re.compile(r'[\u0600-\u06FF\u0400-\u04FF\u3040-\u30FF\u4E00-\u9FFF\-/]')

def normaliser_et_nettoyer(texte):
    if pd.isna(texte): return texte
    texte = re.sub(r'^\s*-\s*', '', str(texte))          # tiret en début
    texte = pattern_exotiques.sub('', texte)               # caractères non-latin
    texte = unidecode.unidecode(texte)                     # <-- MODIFICATION ICI : accents → ASCII
    return re.sub(r'\s+', ' ', texte).strip()

def nettoyer_resume(texte):
    if pd.isna(texte): return texte
    texte = re.sub(r'<.*?>', '', str(texte))               # balises HTML
    texte = re.sub(r'&\w+;', '', texte)                    # codes HTML
    m = re.search(r'[A-Z]', texte)
    if m: texte = texte[m.start():]
    return texte.strip()

print("--- Nettoyage des données ---")
cols_exclure = ['Latitude', 'Longitude', 'HalID']
for col in dashboard_df.select_dtypes(include=['object', 'string']).columns:
    if col not in cols_exclure:
        dashboard_df[col] = dashboard_df[col].apply(normaliser_et_nettoyer)

if 'Resume' in dashboard_df.columns:
    dashboard_df['Resume'] = dashboard_df['Resume'].apply(nettoyer_resume)

dashboard_df = dashboard_df.fillna("")

# --- Exports ---
print("--- Export ---")
dashboard_df.to_excel(FILE_FINAL,       index=False)
dashboard_df.to_excel("dashboard.xlsx", index=False)
dashboard_df.to_csv("Copublis_dashboard.csv", index=False)

lat_series  = dashboard_df.get('Latitude', pd.Series(dtype=str)).replace("", pd.NA)
mask_found  = lat_series.notna()
mask_miss   = ~mask_found

dashboard_df[mask_found].to_excel(FILE_FOUND, index=False)
if mask_miss.sum() > 0:
    dashboard_df[mask_miss].to_excel(FILE_MISSING, index=False)

print(f"\n📄 {FILE_FINAL}")
print(f"📊 dashboard.xlsx / Copublis_dashboard.csv")
print(f"✅ {FILE_FOUND}")
if mask_miss.sum() > 0:
    print(f"❌ {FILE_MISSING} ({mask_miss.sum()} lignes)")

--- Nettoyage des données ---
--- Export ---

📄 copublications_Inria_2018-2024_avec_villes.xlsx
📊 dashboard.xlsx / Copublis_dashboard.csv
✅ Resultat_Villes_TROUVEES.xlsx
❌ Resultat_Villes_MANQUANTES.xlsx (17 lignes)


In [10]:
# =========================================================
#  BILAN FINAL
# =========================================================
total    = len(dashboard_df)
lat_col  = dashboard_df.get('Latitude', pd.Series(dtype=str)).replace("", pd.NA)
n_geo    = int(lat_col.notna().sum())
n_miss   = total - n_geo

print("\n" + "="*50)
print("📊 BILAN FINAL DU PIPELINE")
print("="*50)
print(f"\n📌 Lignes totales            : {total}")
print(f"✅ Lignes géolocalisées      : {n_geo} ({n_geo/total*100:.1f}%)")
print(f"❌ Sans coordonnées          : {n_miss} ({n_miss/total*100:.1f}%)")

if 'HalID' in dashboard_df.columns:
    n_publis = dashboard_df['HalID'].replace("", pd.NA).nunique()
    print(f"\n📚 Publications uniques      : {n_publis}")

print("\n🔍 Détail des méthodes :")
print(f"   Aurehal (dictionnaire)       : {stats['aurehal']}")
print(f"   Nominatim (Adresse+Pays)     : {stats['nominatim_addr']}")
print(f"   Nominatim (Organisme+Pays)   : {stats['nominatim_org']}")
print(f"   Nominatim (Ville+Pays)       : {stats['nominatim_ville']}")
print(f"   Nominatim (Pays seul)        : {stats['nominatim_pays']}")
print(f"   Échec total                  : {stats['echec']}")
print("="*50)


📊 BILAN FINAL DU PIPELINE

📌 Lignes totales            : 79530
✅ Lignes géolocalisées      : 79513 (100.0%)
❌ Sans coordonnées          : 17 (0.0%)

📚 Publications uniques      : 14354

🔍 Détail des méthodes :
   Aurehal (dictionnaire)       : 79513
   Nominatim (Adresse+Pays)     : 0
   Nominatim (Organisme+Pays)   : 0
   Nominatim (Ville+Pays)       : 0


KeyError: 'nominatim_pays'

In [ ]:
# =========================================================
#  EXPORT DASHBOARD DRI (avec colonnes Top Copubliant)
# =========================================================
df_dri = dashboard_df.copy()

if 'Nom_org_Top_copubliant' in df.columns:
    df_dri['Top_Copubliant'] = df['Nom_org_Top_copubliant'].values
else:
    df_dri['Top_Copubliant'] = ""

for col_top in ['id_Aurehal_Top_copubliant', 'id_Aurehal_org_Top_copubliant']:
    if col_top in df.columns:
        df_dri['ID_Aurehal_Top'] = df[col_top].values
        break
else:
    df_dri['ID_Aurehal_Top'] = ""

colonnes = list(df_dri.columns)
for c in ['Top_Copubliant', 'ID_Aurehal_Top']:
    if c in colonnes: colonnes.remove(c)

if 'ID_Aurehal' in colonnes:
    idx = colonnes.index('ID_Aurehal') + 1
    colonnes.insert(idx,     'Top_Copubliant')
    colonnes.insert(idx + 1, 'ID_Aurehal_Top')
else:
    colonnes += ['Top_Copubliant', 'ID_Aurehal_Top']

df_dri = df_dri[colonnes].fillna("")
df_dri.to_excel("dashboard_dri.xlsx", index=False)
print("✅ dashboard_dri.xlsx généré.")

---
## Mise à jour du dictionnaire Aurehal

Cette cellule enrichit `ID_Aurehal_Ville_Etat_Latitude_Longitude.xlsx` avec les résultats Nominatim.

**Effet sur les prochaines exécutions :** les IDs déjà résolus par Nominatim seront trouvés dès l'Étape 1, sans aucune requête réseau. Le pipeline devient de plus en plus rapide à chaque passage.

**Règles d'alimentation :**
- On n'écrase **jamais** une entrée existante (les données Aurehal originales ont priorité)
- On n'ajoute que les résultats Nominatim **avec coordonnées valides**
- On n'ajoute pas les résultats obtenus uniquement via `Pays seul` (trop imprécis)

In [11]:
# =========================================================
#  ALIMENTATION DU DICTIONNAIRE AUREHAL
#
#  Ajoute dans FILE_REF_ID les résultats Nominatim fiables,
#  pour que les prochaines exécutions les trouvent
#  directement à l'Étape 1 sans refaire de requêtes réseau.
# =========================================================
import pandas as pd

print(f"--- Mise à jour du dictionnaire : {FILE_REF_ID} ---")

# --- 1. Charger le dictionnaire existant ---
df_dict = pd.read_excel(FILE_REF_ID)
ids_deja_connus = set(df_dict[COL_REF_ID].dropna().astype(str))
print(f"   Entrées actuelles dans le dictionnaire : {len(ids_deja_connus)}")

# --- 2. Construire les nouvelles entrées depuis le cache Nominatim ---
# Méthodes jugées suffisamment fiables pour alimenter le dictionnaire
METHODES_FIABLES = {
    "Nominatim: Adresse+Pays",
    "Nominatim: Adresse seule",
    "Nominatim: Organisme+Pays",
    "Nominatim: Ville+Pays",
}
# "Nominatim: Pays seul" est exclu : trop imprécis (renvoie la capitale)

nouvelles_entrees = []
for aurehal_id, (lat, lon, ville, methode) in geocode_cache.items():
    # Ignorer les échecs
    if lat is None or lon is None:
        continue
    # Ignorer les méthodes trop imprécises
    if methode not in METHODES_FIABLES:
        continue
    # Ne jamais écraser une entrée existante
    if str(aurehal_id) in ids_deja_connus:
        continue

    # Récupérer le code état US si disponible (depuis df)
    mask_id = df[COL_MAIN_ID].astype(str) == str(aurehal_id)
    state_code = None
    if mask_id.any() and COL_INTERNAL_STATE in df.columns:
        state_code = df.loc[mask_id, COL_INTERNAL_STATE].dropna().head(1)
        state_code = state_code.iloc[0] if not state_code.empty else None

    nouvelles_entrees.append({
        COL_REF_ID        : aurehal_id,
        COL_REF_CITY      : ville,
        COL_REF_LAT       : lat,
        COL_REF_LON       : lon,
        COL_REF_STATE_SRC : state_code,
        "Source"          : methode,   # traçabilité
    })

print(f"   Nouvelles entrées à ajouter : {len(nouvelles_entrees)}")

# --- 3. Fusionner et sauvegarder ---
if nouvelles_entrees:
    df_nouvelles = pd.DataFrame(nouvelles_entrees)

    # Aligner les colonnes : on garde les colonnes du dictionnaire existant
    # et on complète avec NaN pour les colonnes absentes dans df_nouvelles
    for col in df_dict.columns:
        if col not in df_nouvelles.columns:
            df_nouvelles[col] = None

    # Concaténer en conservant uniquement les colonnes du dictionnaire original
    # (+ "Source" si on veut la traçabilité)
    cols_finales = list(df_dict.columns)
    if "Source" not in cols_finales:
        cols_finales.append("Source")

    df_nouvelles = df_nouvelles[[c for c in cols_finales if c in df_nouvelles.columns]]

    df_dict_enrichi = pd.concat([df_dict, df_nouvelles], ignore_index=True)

    # Sauvegarde (écrase le fichier existant)
    df_dict_enrichi.to_excel(FILE_REF_ID, index=False)

    print(f"\n✅ Dictionnaire mis à jour : {FILE_REF_ID}")
    print(f"   Avant : {len(ids_deja_connus)} entrées")
    print(f"   Après : {len(df_dict_enrichi)} entrées")
    print(f"   Ajout : +{len(nouvelles_entrees)} nouvelles entrées")
    print(f"\n💡 Ces {len(nouvelles_entrees)} IDs seront résolus localement lors de la prochaine exécution.")
else:
    print("ℹ️  Aucune nouvelle entrée fiable à ajouter au dictionnaire.")

--- Mise à jour du dictionnaire : ID_Aurehal_Ville_Etat_Latitude_Longitude.xlsx ---
   Entrées actuelles dans le dictionnaire : 7576
   Nouvelles entrées à ajouter : 0
ℹ️  Aucune nouvelle entrée fiable à ajouter au dictionnaire.
